# ParasiteID - Nits vs Other image classifier

Binary image classification comparing **MobileNetV3Large**, **EfficientNetB0**, and **ResNet50V2**.

* Positive class: **nits** (head-lice eggs) - folders `nits/`, `nits/Hair nits/`, `nits/Hair nits train/`
* Negative class: **other** - folders `Dandruff/`, `Healthy Scalp/`, `lice/`

Transfer learning (ImageNet weights), two-phase training (freeze head -> fine-tune top layers),
trained on CPU at 224x224. Run the cells top to bottom.

## 1. Environment & configuration

In [ ]:
import os, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
print("CPUs:", os.cpu_count())

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# --- CPU threads ---
N_THREADS = os.cpu_count() or 8
tf.config.threading.set_intra_op_parallelism_threads(N_THREADS)
tf.config.threading.set_inter_op_parallelism_threads(N_THREADS)

# --- Config ---
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 15
EPOCHS_FINETUNE = 10
FINETUNE_LAYERS = 30
AUTOTUNE = tf.data.AUTOTUNE

DATA_DIR = Path(".")
OUT_MODELS = Path("models")
OUT_RESULTS = Path("results")
OUT_LOGS = Path("logs")
for d in (OUT_MODELS, OUT_RESULTS, OUT_LOGS):
    d.mkdir(exist_ok=True)

CLASS_DIRS = {
    "nits":  ["nits"],
    "other": ["Dandruff", "Healthy Scalp", "lice"],
}

MODELS = {
    "MobileNetV3Large": dict(ctor=tf.keras.applications.MobileNetV3Large,
                             preprocess=tf.keras.applications.mobilenet_v3.preprocess_input),
    "EfficientNetB0":   dict(ctor=tf.keras.applications.EfficientNetB0,
                             preprocess=tf.keras.applications.efficientnet.preprocess_input),
    "ResNet50V2":       dict(ctor=tf.keras.applications.ResNet50V2,
                             preprocess=tf.keras.applications.resnet_v2.preprocess_input),
}
print("Models registered:", list(MODELS))

ModuleNotFoundError: No module named 'tensorflow'

## 2. Index the images

Recursively scan each source folder, validate/read each image, map it to its binary label, and remove content duplicates.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp"}

def iter_images(root):
    root = Path(root)
    if root.is_file():
        if root.suffix.lower() in IMG_EXTS:
            yield root
        return
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p

def file_hash(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def probe(path):
    """Return (width, height) or (None, None) if the image cannot be read."""
    try:
        with Image.open(path) as im:
            w, h = im.size
            im.convert("RGB")
        return w, h
    except Exception:
        return None, None

records = []
for label, folders in CLASS_DIRS.items():
    for folder in folders:
        n = 0
        for p in iter_images(DATA_DIR / folder):
            w, h = probe(p)
            if w is None:
                print("  skip (unreadable):", p)
                continue
            records.append({"filepath": str(p), "label": label, "width": w, "height": h})
            n += 1
        print(f"{label:5s} <- {folder:25s} {n:5d} images")

df = pd.DataFrame(records)
print("\nRaw images:", len(df))

df["hash"] = df["filepath"].map(file_hash)
before = len(df)
df = df.drop_duplicates("hash").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate images -> {len(df)} unique")

df["y"] = (df["label"] == "nits").astype(int)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
df.head()

## 3. Quick EDA

In [ ]:
print(df["label"].value_counts(), "\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["label"].value_counts().plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Class balance"); axes[0].set_ylabel("images"); axes[0].tick_params(axis="x", rotation=0)
axes[1].scatter(df["width"], df["height"], s=6, alpha=0.3)
axes[1].set_xlabel("width"); axes[1].set_ylabel("height"); axes[1].set_title("Image dimensions")
plt.tight_layout(); plt.savefig(OUT_RESULTS / "eda.png", dpi=120); plt.show()

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for row, label in enumerate(["nits", "other"]):
    subset = df[df["label"] == label].sample(6, random_state=SEED)
    for col, (_, r) in enumerate(subset.iterrows()):
        ax = axes[row, col]
        ax.imshow(Image.open(r["filepath"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
        ax.axis("off")
        if col == 0:
            ax.set_title(label, fontsize=12)
plt.tight_layout(); plt.show()

## 4. Train / validation / test split

Stratified 70/15/15 on the binary label. The same split (saved to `results/splits.json`) is used by every model so the comparison is fair.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["y"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["y"], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name:5s}", d["label"].value_counts().to_dict())

splits = {name: d["filepath"].tolist() for name, d in
          [("train", train_df), ("val", val_df), ("test", test_df)]}
(OUT_RESULTS / "splits.json").write_text(json.dumps(splits, indent=2))

cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["y"].values)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print("class_weight:", class_weight)

## 5. `tf.data` pipelines

Decode -> resize -> augment (train only) -> model-specific preprocessing. Each model needs its own preprocessing function, so pipelines are built per model inside `make_ds`.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")

def decode_img(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.ensure_shape(img, [None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), label

def make_ds(paths, labels, preprocess, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_img, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

## 6. Model factory

Shared head: `GlobalAveragePooling2D -> Dropout -> Dense(1, sigmoid)` on top of each frozen ImageNet backbone.

In [ ]:
def build_model(name):
    base = MODELS[name]["ctor"](include_top=False, weights="imagenet",
                                input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs, outputs, name=name), base

def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=["accuracy",
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall"),
                 tf.keras.metrics.AUC(name="auc")],
    )

## 7. Train (two-phase transfer learning)

Phase 1 freezes the backbone and trains the head. Phase 2 unfreezes the top layers and fine-tunes at a low LR. Same callbacks, epochs, augmentation and class weights for every model.

In [ ]:
def plot_history(name, hist):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, metric in zip(axes, ["accuracy", "loss", "auc"]):
        ax.plot(hist["epoch"], hist[metric], label="train")
        if f"val_{metric}" in hist:
            ax.plot(hist["epoch"], hist[f"val_{metric}"], label="val")
        ax.axvline(EPOCHS_HEAD + 0.5, color="gray", ls=":", lw=1)
        ax.set_title(f"{name}: {metric}"); ax.set_xlabel("epoch"); ax.legend()
    plt.tight_layout(); plt.show()

def train_model(name, verbose=1):
    tf.keras.backend.clear_session()
    print(f"\n{'=' * 60}\nTraining {name}\n{'=' * 60}")
    model, base = build_model(name)
    preprocess = MODELS[name]["preprocess"]

    train_ds = make_ds(train_df["filepath"].values, train_df["y"].values, preprocess, True)
    val_ds = make_ds(val_df["filepath"].values, val_df["y"].values, preprocess, False)

    ckpt_path = str(OUT_MODELS / f"{name}.keras")
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_auc",
                                           mode="max", save_best_only=True, verbose=verbose),
        tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                         patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                             patience=3, min_lr=1e-7, verbose=verbose),
    ]

    t0 = time.time()
    compile_model(model, 1e-3)
    hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                      class_weight=class_weight, callbacks=callbacks, verbose=verbose)

    base.trainable = True
    for layer in base.layers[:-FINETUNE_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    compile_model(model, 1e-5)
    hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
                      class_weight=class_weight, callbacks=callbacks, verbose=verbose)
    elapsed = time.time() - t0

    h1 = pd.DataFrame(hist1.history); h1["phase"] = "head"
    h2 = pd.DataFrame(hist2.history); h2["phase"] = "finetune"
    hist = pd.concat([h1, h2], ignore_index=True)
    hist.insert(0, "epoch", np.arange(1, len(hist) + 1))
    hist.to_csv(OUT_RESULTS / f"{name}_history.csv", index=False)

    val_auc = max(list(hist1.history.get("val_auc", [0.0])) +
                  list(hist2.history.get("val_auc", [0.0])))
    meta = dict(model=name,
                params=int(model.count_params()),
                size_mb=round(os.path.getsize(ckpt_path) / 1e6, 1),
                train_seconds=round(elapsed, 1),
                val_auc=float(val_auc),
                epochs_ran=int(len(hist)))
    (OUT_RESULTS / f"{name}_meta.json").write_text(json.dumps(meta, indent=2))
    print(f"{name}: {elapsed / 60:.1f} min | {meta['params'] / 1e6:.2f}M params "
          f"| {meta['size_mb']} MB | best val_auc={meta['val_auc']:.4f}")
    return model, hist

### Run training

Set `MODELS_TO_TRAIN` to one model (faster) or all three for the full comparison. Checkpoints and histories are written per model, so you can stop and resume.

In [ ]:
MODEL_TO_RUN = "MobileNetV3Large"     # quickest on CPU
MODELS_TO_TRAIN = list(MODELS)

for name in MODELS_TO_TRAIN:
    _, hist = train_model(name)
    plot_history(name, hist)

## 8. Evaluate on the held-out test set

In [ ]:
def evaluate_model(name):
    model = tf.keras.models.load_model(OUT_MODELS / f"{name}.keras")
    preprocess = MODELS[name]["preprocess"]
    test_ds = make_ds(test_df["filepath"].values, test_df["y"].values, preprocess, False)

    y_true = test_df["y"].values
    y_prob = model.predict(test_ds, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=["other", "nits"], digits=4))

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn) / cm.sum()
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    auc = roc_auc_score(y_true, y_prob)
    print(f"acc={acc:.4f} | nits precision={prec:.4f} recall={rec:.4f} "
          f"f1={f1:.4f} | auc={auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=13)
    axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(["other", "nits"]); axes[0].set_yticklabels(["other", "nits"])
    axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true")
    axes[0].set_title(f"{name} confusion matrix")

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr, tpr, label=f"AUC={auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
    axes[1].set_title(f"{name} ROC"); axes[1].legend()
    plt.tight_layout(); plt.savefig(OUT_RESULTS / f"{name}_eval.png", dpi=120); plt.show()

    result = dict(model=name, test_acc=acc, test_precision=prec, test_recall=rec,
                  test_f1=f1, test_auc=auc)
    (OUT_RESULTS / f"{name}_eval.json").write_text(json.dumps(result, indent=2))
    return result

In [ ]:
for name in MODELS_TO_TRAIN:
    evaluate_model(name)

## 9. Model comparison

In [ ]:
def collect_results():
    rows = []
    for name in MODELS:
        meta_p = OUT_RESULTS / f"{name}_meta.json"
        eval_p = OUT_RESULTS / f"{name}_eval.json"
        if meta_p.exists() and eval_p.exists():
            rows.append({**json.loads(meta_p.read_text()),
                         **json.loads(eval_p.read_text())})
    return pd.DataFrame(rows)

summary = collect_results()
if summary.empty:
    print("No evaluated models yet - train and evaluate at least one model above.")
else:
    summary = summary.sort_values("test_f1", ascending=False).reset_index(drop=True)
    cols = ["model", "params", "size_mb", "train_seconds", "val_auc",
            "test_acc", "test_precision", "test_recall", "test_f1", "test_auc"]
    display(summary[cols])
    summary.to_csv(OUT_RESULTS / "model_comparison.csv", index=False)
    best = summary.iloc[0]
    print(f"\nBest model by test F1: {best['model']} "
          f"(F1={best['test_f1']:.4f}, AUC={best['test_auc']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name in MODELS:
    p = OUT_RESULTS / f"{name}_history.csv"
    if not p.exists():
        continue
    h = pd.read_csv(p)
    axes[0].plot(h["epoch"], h["val_accuracy"], marker="o", ms=3, label=name)
    axes[1].plot(h["epoch"], h["val_loss"], marker="o", ms=3, label=name)
for ax, title in zip(axes, ["Validation accuracy", "Validation loss"]):
    ax.axvline(EPOCHS_HEAD + 0.5, color="gray", ls=":", lw=1)
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.savefig(OUT_RESULTS / "comparison_curves.png", dpi=120); plt.show()